In [ ]:
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import xgboost as xgb
from xgboost import callback

# ============================================================
# 1) Chargement JSONL
# ============================================================

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf8") as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

train_df = load_jsonl("../../data/train.jsonl")
test_df = load_jsonl("../../data/kaggle_test.jsonl")

y = train_df["label"]

print("Train:", train_df.shape)
print("Test:", test_df.shape)

# ============================================================

# ------------------------------------------------------------
# 1) IDENTIFIER colonnes qui contiennent dict ou list
# ------------------------------------------------------------
def is_dict_or_list(x):
    return isinstance(x, dict) or isinstance(x, list)

dict_cols_train = [
    c for c in train_df.columns
    if train_df[c].apply(is_dict_or_list).any()
]
dict_cols_test = [
    c for c in test_df.columns
    if test_df[c].apply(is_dict_or_list).any()
]

print("Colonnes à flatten :", dict_cols_train)


# ------------------------------------------------------------
# 2) Fonction robuste pour flatten dict + list
# ------------------------------------------------------------
def robust_expand_column(series, prefix):
    """
    Transforme une colonne contenant dict/list en colonnes scalables.
    """
    expanded_rows = []

    for val in series:
        if isinstance(val, dict):
            expanded_rows.append(val)

        elif isinstance(val, list):
            row = {"len": len(val)}
            for i, item in enumerate(val):
                if isinstance(item, (str, int, float, bool)):
                    row[f"item_{i}"] = item
                else:
                    row[f"item_{i}"] = str(item)
            expanded_rows.append(row)

        else:
            expanded_rows.append({})

    df_expanded = pd.json_normalize(expanded_rows)
    df_expanded = df_expanded.add_prefix(prefix + ".")
    return df_expanded


# ------------------------------------------------------------
# 3) Expansion identique train / test
# ------------------------------------------------------------
def expand_dict_and_list_columns(df, dict_cols):
    df_new = df.copy()
    for col in dict_cols:
        expanded = robust_expand_column(df_new[col], col)
        df_new = pd.concat([df_new.drop(columns=[col]), expanded], axis=1)
    return df_new

train_df = expand_dict_and_list_columns(train_df, dict_cols_train)
test_df  = expand_dict_and_list_columns(test_df , dict_cols_test)


# ------------------------------------------------------------
# 4) REALIGNEMENT colonnes train/test
# ------------------------------------------------------------
train_cols = set(train_df.columns)
test_cols  = set(test_df.columns)

missing = train_cols - test_cols
extra   = test_cols - train_cols

for col in missing:
    test_df[col] = 0

test_df = test_df.drop(columns=list(extra))

train_df = train_df.sort_index(axis=1)
test_df  = test_df.sort_index(axis=1)

print("Colonnes finales :", len(train_df.columns))


# ------------------------------------------------------------
# 5) EXTRACTION FEATURES NUMÉRIQUES + SCALING ROBUSTE
# ------------------------------------------------------------
exclude = {"ID", "label", "text"}
num_cols = [c for c in train_df.columns if c not in exclude]

# Conversion en numérique
train_num = train_df[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
test_num  = test_df[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

# StandardScaler
scaler = StandardScaler()
X_num_train = scaler.fit_transform(train_num)
X_num_test  = scaler.transform(test_num)

# Clip pour éviter les valeurs trop grandes / infinies
upper_limit = 1e15
lower_limit = -1e15
X_num_train = np.clip(X_num_train, lower_limit, upper_limit)
X_num_test  = np.clip(X_num_test,  lower_limit, upper_limit)

print("Num features OK — dims :", X_num_train.shape, X_num_test.shape)

# ------------------------------------------------------------
# 6) TF-IDF + SVD
# ------------------------------------------------------------
text_col = "text"

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=3)
X_tfidf_train = tfidf.fit_transform(train_df[text_col])
X_tfidf_test  = tfidf.transform(test_df[text_col])

svd = TruncatedSVD(n_components=256)
X_tfidf_train = svd.fit_transform(X_tfidf_train)
X_tfidf_test  = svd.transform(X_tfidf_test)

# Clip TF-IDF SVD si nécessaire
X_tfidf_train = np.clip(X_tfidf_train, lower_limit, upper_limit)
X_tfidf_test  = np.clip(X_tfidf_test,  lower_limit, upper_limit)

# ------------------------------------------------------------
# 7) CONCAT TF-IDF + FEATURES NUMÉRIQUES
# ------------------------------------------------------------
X_train_0 = np.hstack([X_tfidf_train, X_num_train])
X_test    = np.hstack([X_tfidf_test , X_num_test ])

# Vérification finale
print("Final dim:", X_train_0.shape, X_test.shape)
print("Min/Max X_train_0:", np.min(X_train_0), np.max(X_train_0))
print("Min/Max X_test:", np.min(X_test), np.max(X_test))


In [ ]:
# ============================================================
# 1) Train/Validation split
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import xgboost as xgb
import numpy as np


X_trained, X_val, y_train, y_val = train_test_split(
    X_train_0, y, test_size=0.1, random_state=42
)


# ============================================================
# 2) XGBoost (compatible toutes versions)
# ============================================================
dtrain = xgb.DMatrix(X_trained, label=y_train)
dval   = xgb.DMatrix(X_val, label=y_val)
# dtest = xgb.DMatrix(X_test)

params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "eta": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
}

# On peut surveiller la validation tous les 50 rounds
xgb_model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=1000,
    evals=[(dtrain, "train"), (dval, "val")],
    early_stopping_rounds=50,
    verbose_eval=50  # <-- affiche l’accuracy/logloss tous les 50 rounds
)

# ============================================================
# 3) Accuracy finale sur la validation
# ============================================================
y_val_pred = (xgb_model.predict(dval) >= 0.5).astype(int)
acc = accuracy_score(y_val, y_val_pred)
print(f"[XGBOOST] Validation accuracy: {acc:.4f}")


In [ ]:
print(X_test)
import numpy as np

# Vérification des NaN
num_nan = np.isnan(X_test).sum()
# Vérification des infinis positifs
num_posinf = np.isposinf(X_test).sum()
# Vérification des infinis négatifs
num_neginf = np.isneginf(X_test).sum()
# Vérification que toutes les valeurs sont finies
all_finite = np.isfinite(X_test).all()

print(f"X_test contient {num_nan} NaN")
print(f"X_test contient {num_posinf} +inf")
print(f"X_test contient {num_neginf} -inf")
print(f"Toutes les valeurs sont finies ? {all_finite}")


In [ ]:
if X_test.shape[1] != X_trained.shape[1]:
    print("Mismatch ! Realignement nécessaire.")
else:
    print("ok")

# Vérifier si l'ordre des colonnes numériques est identique
same_order = list(train_num.columns) == list(test_num.columns)
print("L'ordre des colonnes numériques est identique ?", same_order)


In [ ]:
import numpy as np
import pandas as pd

# Vérifier NaN
n_nan = np.isnan(X_test).sum()
print(f"X_test contient {n_nan} NaN")

# Vérifier +inf et -inf
n_posinf = np.isposinf(X_test).sum()
n_neginf = np.isneginf(X_test).sum()
print(f"X_test contient {n_posinf} +inf")
print(f"X_test contient {n_neginf} -inf")

# Vérifier si toutes les valeurs sont finies
all_finite = np.isfinite(X_test).all()
print("Toutes les valeurs sont finies ?", all_finite)

# Vérifier types de données
print("Type des données :", X_test.dtype)

# Vérifier les valeurs extrêmes
print("Valeurs min/max :", np.min(X_test), np.max(X_test))

# Si certaines valeurs sont très grandes (>1e10 par exemple)
threshold = 1e10
too_large = np.sum(np.abs(X_test) > threshold)
print(f"Nombre de valeurs > {threshold} :", too_large)

# Vérifier shape
print("Shape de X_test :", X_test.shape)

# Optionnel : afficher les indices des valeurs problématiques
if too_large > 0:
    idx_large = np.where(np.abs(X_test) > threshold)
    print("Indices des valeurs trop grandes :", idx_large)

dtest = xgb.DMatrix(X_test)

In [ ]:
# Créer le DMatrix propre
# Si X_test est un ndarray, assure-toi que train et test ont les mêmes colonnes



##
y_test_pred = (xgb_model.predict(dtest) >= 0.5).astype(int)

# ============================================================
# 8️⃣ Création du fichier submission.csv
# ============================================================
if "ID" in test_df.columns:
    submission = pd.DataFrame({"ID": test_df["ID"], "Prediction": y_test_pred})
elif "challenge_id" in test_df.columns:
    submission = pd.DataFrame({"ID": test_df["challenge_id"], "Prediction": y_test_pred})
else:
    raise ValueError("Aucune colonne ID valide trouvée dans test_df !")


submission.to_csv("submission_xgboost.csv", index=False)
print("Fichier 'submission_xgboost.csv' généré ✔️")